# TP4 — De Cassandra à Spark : analyse des disponibilités Vélib'

**Question métier :** quels niveaux de disponibilité observe-t-on parmi les stations Vélib', et combien de stations sont presque vides ?

Les données viennent de `velib_cluster.stations_velib`, créée au TP3. La table contient une ligne par station. Attention : `vatiques_disponibles` est le nom réel de la colonne dans CQL; elle représente les vélos disponibles. Le scénario de panne du TP3 a ajouté une ligne de test `99999` (et peut-être `99998`), exclue de l'analyse.

Le connecteur Cassandra se télécharge depuis Maven au premier démarrage de Spark. Garder cette session active jusqu'à la fin de l'observation dans Spark UI.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

KEYSPACE = "velib_cluster"
TABLE = "stations_velib"
CONNECTOR = "com.datastax.spark:spark-cassandra-connector_2.12:3.5.1"

spark = (
    SparkSession.builder
    .appName("TP4-Cassandra-Spark-Velib")
    .config("spark.jars.packages", CONNECTOR)
    .config("spark.cassandra.connection.host", "cass1")
    .config("spark.cassandra.connection.port", "9042")
    .config("spark.sql.shuffle.partitions", "3")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version)
print("Application ID:", spark.sparkContext.applicationId)
print("Spark UI:", spark.sparkContext.uiWebUrl)

Spark: 3.5.0
Application ID: local-1791454568241
Spark UI: http://49043226d8e1:4040


## 1. Lecture de Cassandra

Le connecteur lit la table métier et la transforme en DataFrame Spark. L'hôte `cass1` est résolu par Docker car le conteneur Jupyter partage le réseau `cass-net` avec le cluster Cassandra.

In [2]:
df = (
    spark.read
    .format("org.apache.spark.sql.cassandra")
    .options(keyspace=KEYSPACE, table=TABLE)
    .load()
)

print(f"Keyspace : {KEYSPACE}\nTable : {TABLE}")
df.printSchema()
df.show(5, truncate=False)
print("Nombre de lignes lues :", df.count())

Keyspace : velib_cluster
Table : stations_velib
root
 |-- station_id: string (nullable = false)
 |-- bornes_disponibles: integer (nullable = true)
 |-- capacite: integer (nullable = true)
 |-- derniere_mise_a_jour: timestamp (nullable = true)
 |-- nom_station: string (nullable = true)
 |-- vatiques_disponibles: integer (nullable = true)



+----------+------------------+--------+--------------------+----------------------------+--------------------+
|station_id|bornes_disponibles|capacite|derniere_mise_a_jour|nom_station                 |vatiques_disponibles|
+----------+------------------+--------+--------------------+----------------------------+--------------------+
|17041     |27                |36      |2026-10-07 08:28:04 |Guersant - Gouvion-Saint-Cyr|5                   |
|5028      |1                 |28      |2026-10-07 08:26:53 |Censier - Jardin des Plantes|25                  |
|11023     |58                |65      |2026-10-07 08:28:43 |Saint-Maur - Roquette       |6                   |
|17043     |28                |39      |2026-10-07 08:27:21 |Belidor - Gouvion-Saint-Cyr |8                   |
|13118     |6                 |63      |2026-10-07 08:27:45 |Regnault - Patay            |56                  |
+----------+------------------+--------+--------------------+----------------------------+--------------

Nombre de lignes lues : 101


Les colonnes utiles sont l'identifiant et le nom de station, la capacité, le nombre de vélos disponibles et les bornes libres. On retire les lignes synthétiques du TP3 et les capacités nulles ou non positives avant de calculer un taux.

In [3]:
df_stations = (
    df.filter(~F.col("station_id").isin("99998", "99999"))
      .filter(F.col("capacite").isNotNull() & (F.col("capacite") > 0))
      .select(
          "station_id", "nom_station", "capacite",
          "vatiques_disponibles", "bornes_disponibles",
          "derniere_mise_a_jour"
      )
)

print("Stations métier retenues :", df_stations.count())
df_stations.show(5, truncate=False)

Stations métier retenues : 100


+----------+-----------------------------------------+--------+--------------------+------------------+--------------------+
|station_id|nom_station                              |capacite|vatiques_disponibles|bornes_disponibles|derniere_mise_a_jour|
+----------+-----------------------------------------+--------+--------------------+------------------+--------------------+
|17026     |Jouffroy d'Abbans - Wagram               |40      |10                  |29                |2026-10-07 08:27:14 |
|92002     |Théâtre des Amandiers - Palais des sports|23      |17                  |4                 |2026-10-07 08:25:21 |
|18011     |Pajol - Place Herbert                    |39      |1                   |35                |2026-10-07 08:28:20 |
|10042     |Enghien - Faubourg Poissonnière          |33      |29                  |2                 |2026-10-07 08:27:34 |
|10161     |Gare de l'Est - Faubourg-Saint-Martin    |32      |1                   |29                |2026-10-07 08:28:53 |


## 2. Filtres métier et indicateur calculé

Deux filtres répondent à des situations opérationnelles opposées : stations presque vides (au plus cinq vélos) et stations très approvisionnées (plus de vingt vélos). Le taux rapporte les vélos disponibles à la capacité; les niveaux facilitent la comparaison entre stations de tailles différentes.

In [4]:
stations_presque_vides = df_stations.filter(F.col("vatiques_disponibles") <= 5)
stations_bien_approvisionnees = df_stations.filter(F.col("vatiques_disponibles") > 20)

print("Au plus 5 vélos disponibles :", stations_presque_vides.count())
stations_presque_vides.select("station_id", "nom_station", "capacite", "vatiques_disponibles").show(10, truncate=False)
print("Plus de 20 vélos disponibles :", stations_bien_approvisionnees.count())
stations_bien_approvisionnees.select("station_id", "nom_station", "capacite", "vatiques_disponibles").show(10, truncate=False)

Au plus 5 vélos disponibles : 34


+----------+-------------------------------------+--------+--------------------+
|station_id|nom_station                          |capacite|vatiques_disponibles|
+----------+-------------------------------------+--------+--------------------+
|9020      |Toudouze - Clauzel                   |21      |3                   |
|18111     |Félix Ziem - Armand Gauthier         |25      |1                   |
|26003     |Argenteuil - Voltaire                |16      |3                   |
|19207     |Jean Jaurès - Bouret                 |32      |4                   |
|18011     |Pajol - Place Herbert                |39      |1                   |
|10161     |Gare de l'Est - Faubourg-Saint-Martin|32      |1                   |
|14108     |Le Brix et Mesmin - Jourdan          |21      |1                   |
|31102     |Boissière - Etienne Dolet            |21      |0                   |
|10107     |Gare du Nord - Hôpital Lariboisière  |49      |3                   |
|18046     |Etienne Jodelle 

Plus de 20 vélos disponibles : 19


+----------+----------------------------------+--------+--------------------+
|station_id|nom_station                       |capacite|vatiques_disponibles|
+----------+----------------------------------+--------+--------------------+
|12151     |Gare de Lyon - Place Louis Armand |60      |42                  |
|7003      |Square Boucicaut                  |60      |58                  |
|42021     |Raspail - Manufacture des Oeillets|32      |26                  |
|14111     |Cassini - Denfert-Rochereau       |25      |21                  |
|8031      |Jean Mermoz - Champs-Elysées      |49      |43                  |
|2017      |Caire - Dussoubs                  |40      |36                  |
|45003     |11 Novembre 1918 - 8 Mai 1945     |36      |33                  |
|8045      |Place de la Reine Astrid          |58      |39                  |
|16039     |Exelmans - Versailles             |49      |24                  |
|13101     |Croulebarde - Corvisart           |34      |31      

In [5]:
df_analyse = (
    df_stations
    .withColumn(
        "taux_disponibilite",
        F.round(F.col("vatiques_disponibles") / F.col("capacite"), 4)
    )
    .withColumn(
        "niveau_disponibilite",
        F.when(F.col("taux_disponibilite") < 0.10, "critique")
         .when(F.col("taux_disponibilite") < 0.30, "faible")
         .otherwise("correcte")
    )
)
df_analyse.select("station_id", "nom_station", "capacite", "vatiques_disponibles", "taux_disponibilite", "niveau_disponibilite").show(10, truncate=False)

+----------+-----------------------------+--------+--------------------+------------------+--------------------+
|station_id|nom_station                  |capacite|vatiques_disponibles|taux_disponibilite|niveau_disponibilite|
+----------+-----------------------------+--------+--------------------+------------------+--------------------+
|11047     |Amelot - Saint-Sébastien     |30      |9                   |0.3               |correcte            |
|12109     |Mairie du 12ème              |30      |3                   |0.1               |faible              |
|16039     |Exelmans - Versailles        |49      |24                  |0.4898            |correcte            |
|44006     |Concorde - Stalingrad        |35      |5                   |0.1429            |faible              |
|1119      |Cossonnerie - Sébastopol     |24      |13                  |0.5417            |correcte            |
|40016     |Sarrazins - Saussure         |22      |6                   |0.2727            |faibl

## 3. Agrégation : disponibilité par niveau

Cette agrégation répond à la question métier en comptant les stations par niveau, tout en résumant leur taux moyen, leur stock moyen de vélos et la capacité maximale observée. `groupBy` et les agrégations sont des transformations; `show()` déclenche ici leur exécution.

In [6]:
disponibilite_par_niveau = (
    df_analyse.groupBy("niveau_disponibilite")
    .agg(
        F.count("station_id").alias("nombre_stations"),
        F.round(F.avg("taux_disponibilite"), 3).alias("taux_moyen"),
        F.round(F.avg("vatiques_disponibles"), 1).alias("velos_moyens"),
        F.max("capacite").alias("capacite_max")
    )
    .orderBy("niveau_disponibilite")
)
disponibilite_par_niveau.show(truncate=False)

+--------------------+---------------+----------+------------+------------+
|niveau_disponibilite|nombre_stations|taux_moyen|velos_moyens|capacite_max|
+--------------------+---------------+----------+------------+------------+
|correcte            |53             |0.677     |20.9        |63          |
|critique            |23             |0.044     |1.4         |65          |
|faible              |24             |0.182     |6.2         |53          |
+--------------------+---------------+----------+------------+------------+



## 4. Partitions et Lazy Evaluation

Le nombre de partitions contrôle le parallélisme des tâches. `repartition(3)` construit une nouvelle distribution; l'appel seul ne calcule pas les lignes. Spark diffère l'exécution des transformations jusqu'à une action comme `count()`, `show()` ou `collect()`. Cette Lazy Evaluation permet d'optimiser la chaîne complète avant de lancer le travail et d'éviter des calculs inutiles.

In [7]:
print("Partitions du DataFrame lu :", df_stations.rdd.getNumPartitions())
df_3_partitions = df_stations.repartition(3)
print("Partitions après repartition(3) :", df_3_partitions.rdd.getNumPartitions())

pipeline = (
    df_3_partitions
    .filter(F.col("vatiques_disponibles") <= 20)
    .withColumn("taux", F.round(F.col("vatiques_disponibles") / F.col("capacite"), 3))
    .select("station_id", "nom_station", "taux")
)
print("Pipeline défini : aucune action métier n'a encore été appelée sur pipeline.")
pipeline.explain(mode="formatted")

Partitions du DataFrame lu : 21


Partitions après repartition(3) : 3


Pipeline défini : aucune action métier n'a encore été appelée sur pipeline.


== Physical Plan ==
AdaptiveSparkPlan (6)
+- Project (5)
   +- Exchange (4)
      +- Project (3)
         +- Filter (2)
            +- BatchScan stations_velib (1)


(1) BatchScan stations_velib
Output [4]: [station_id#0, capacite#2, nom_station#4, vatiques_disponibles#5]
Cassandra Scan: velib_cluster.stations_velib
 - Cassandra Filters: []
 - Requested Columns: [station_id,capacite,nom_station,vatiques_disponibles]

(2) Filter
Input [4]: [station_id#0, capacite#2, nom_station#4, vatiques_disponibles#5]
Condition : (((vatiques_disponibles#5 <= 20) AND (capacite#2 > 0)) AND NOT station_id#0 IN (99998,99999))

(3) Project
Output [4]: [station_id#0, nom_station#4, capacite#2, vatiques_disponibles#5]
Input [4]: [station_id#0, capacite#2, nom_station#4, vatiques_disponibles#5]

(4) Exchange
Input [4]: [station_id#0, nom_station#4, capacite#2, vatiques_disponibles#5]
Arguments: RoundRobinPartitioning(3), REPARTITION_BY_NUM, [plan_id=358]

(5) Project
Output [3]: [station_id#0, nom_station#4,

In [8]:
# Cette action déclenche l'exécution de la chaîne de transformations ci-dessus.
print("Lignes du pipeline :", pipeline.count())

Lignes du pipeline : 81


## 5. Plan d'exécution et Shuffle

Un `groupBy` redistribue généralement les lignes par clé. Dans le plan physique, repérer `Exchange` (shuffle) et `HashAggregate`. Le plan et les métriques exactes dépendent de l'exécution et des versions installées; ne reporter que ce qui apparaît dans cette session.

In [9]:
disponibilite_par_niveau.explain(mode="formatted")
# L'action lance l'agrégation; consulter ensuite les Jobs et Stages dans Spark UI.
disponibilite_par_niveau.collect()

== Physical Plan ==
AdaptiveSparkPlan (10)
+- Sort (9)
   +- Exchange (8)
      +- HashAggregate (7)
         +- Exchange (6)
            +- HashAggregate (5)
               +- Project (4)
                  +- Project (3)
                     +- Filter (2)
                        +- BatchScan stations_velib (1)


(1) BatchScan stations_velib
Output [3]: [station_id#0, capacite#2, vatiques_disponibles#5]
Cassandra Scan: velib_cluster.stations_velib
 - Cassandra Filters: []
 - Requested Columns: [station_id,capacite,vatiques_disponibles]

(2) Filter
Input [3]: [station_id#0, capacite#2, vatiques_disponibles#5]
Condition : (NOT station_id#0 IN (99998,99999) AND (capacite#2 > 0))

(3) Project
Output [3]: [capacite#2, vatiques_disponibles#5, round((cast(vatiques_disponibles#5 as double) / cast(capacite#2 as double)), 4) AS taux_disponibilite#182]
Input [3]: [station_id#0, capacite#2, vatiques_disponibles#5]

(4) Project
Output [4]: [capacite#2, vatiques_disponibles#5, taux_disponibilite#182

[Row(niveau_disponibilite='correcte', nombre_stations=53, taux_moyen=0.677, velos_moyens=20.9, capacite_max=63),
 Row(niveau_disponibilite='critique', nombre_stations=23, taux_moyen=0.044, velos_moyens=1.4, capacite_max=65),
 Row(niveau_disponibilite='faible', nombre_stations=24, taux_moyen=0.182, velos_moyens=6.2, capacite_max=53)]

## 6. Observation dans Spark UI

1. Laisser la SparkSession active et ouvrir `http://localhost:4040` (ou l'URL affichée par `spark.sparkContext.uiWebUrl`).
2. Dans **Jobs**, ouvrir le Job déclenché par l'action d'agrégation ci-dessus.
3. Relever le nombre de Stages, de Tasks et la durée réellement affichés.
4. Dans le détail du Stage, examiner **Shuffle Read** et **Shuffle Write** lorsqu'ils sont affichés; relier les Tasks aux partitions.
5. Comparer ces observations au `Exchange` du plan physique. Les transformations seules ne créent pas de Job; les actions déclenchent l'exécution.

**À compléter après observation réelle :** Job observé : ___ ; Stages : ___ ; Tasks : ___ ; durée : ___ ; Shuffle Read/Write : ___. Ne pas remplir ces valeurs avant d'avoir consulté l'interface.

Capturez dans `screenshots/` le DataFrame, les partitions et le plan, puis la Spark UI montrant Jobs/Stages/Tasks et les informations Shuffle disponibles.

## Conclusion métier

Sur les 100 stations retenues dans cet instantané, **23 sont en situation critique**, **24 ont une disponibilité faible** et **53 ont une disponibilité correcte**. Le filtre opérationnel `vatiques_disponibles <= 5` identifie 34 stations; le filtre `> 20` en identifie 19. Ces résultats répondent à la question de disponibilité, mais ne décrivent pas une tendance dans le temps.

Le DataFrame source a 21 partitions; `repartition(3)` en produit 3. Les plans physiques affichent `Exchange` pour les redistributions. Les transformations restent paresseuses jusqu'aux actions (`count`, `show`, `collect`), qui déclenchent les Jobs. Compléter les mesures exactes de Jobs, Stages, Tasks et Shuffle après observation dans Spark UI.

Après avoir enregistré les observations et captures Spark UI, la session peut être arrêtée avec `spark.stop()`.